# Decision Tree — v1

**Van Nguyen Thanh Dat — 3124411067 · Nhóm 10 · UCI 697**

## Mục tiêu thí nghiệm

Đánh giá một cây quyết định có độ sâu tối đa 5 và tối thiểu 10 mẫu
huấn luyện mỗi lá. Cây dùng 36 đặc trưng ở dạng số hiện có và được
đánh giá trên cùng 5 fold đã lưu như Dummy và Logistic Regression.

So sánh bằng Macro F1 trung bình trên 5 fold. Dùng dự đoán OOF
để phân tích cây thường nhầm các lớp nào, đặc biệt là Enrolled
và Dropout. Run này chưa chấm tập Test.

In [6]:
from pathlib import Path
import sys


current_dir = Path.cwd()
search_dirs = (current_dir, *current_dir.parents)

FINAL = next(
    folder
    for folder in search_dirs
    if (folder / "src" / "data.py").is_file()
    and (folder / "data" / "raw" / "data.csv").is_file()
)

sys.path.insert(0, str(FINAL))

from src.data import LABELS, load_student_data

print("Thư mục dự án:", FINAL.name)

Thư mục dự án: final


## Mục 2 - Dữ liệu và nhãn:

In [7]:
from IPython.display import display


X, y = load_student_data()

print("Số mẫu:", len(y))
print("Số đặc trưng:", X.shape[1])

label_counts = y.value_counts()
label_counts = label_counts.reindex(LABELS)
label_summary = label_counts.rename_axis("Nhãn").to_frame("Số mẫu")

display(label_summary)

Số mẫu: 4424
Số đặc trưng: 36


,Số mẫu
Nhãn,
Dropout,1421
Enrolled,794
Graduate,2209


## Mục 3 - Nạp bộ chia Train/Test và 5 fold

In [8]:
from src.data import load_splits


train_ids, test_ids, folds, split_manifest = load_splits()

X_train = X.iloc[train_ids]
y_train = y.iloc[train_ids]

print("Số mẫu Train:", len(y_train))
print("Số mẫu Test:", len(test_ids))
print("Số fold:", len(folds))

for fold_number, (fit_positions, valid_positions) in enumerate(folds, start=1):
    print(
        f"Fold {fold_number}: "
        f"{len(fit_positions)} mẫu huấn luyện, "
        f"{len(valid_positions)} mẫu validation"
    )

Số mẫu Train: 3539
Số mẫu Test: 885
Số fold: 5
Fold 1: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 2: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 3: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 4: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 5: 2832 mẫu huấn luyện, 707 mẫu validation


## Mục 4 - Đọc cấu hình và tạo Decision Tree

In [9]:
from runpy import run_path

import yaml


MODEL_ID = "m02_decision_tree"
MODEL_VERSION = "v1"

model_dir = FINAL / "models" / MODEL_ID / MODEL_VERSION
config_path = model_dir / "config.yaml"
train_path = model_dir / "train.py"

config_text = config_path.read_text(encoding="utf-8")
config = yaml.safe_load(config_text)

assert config["model_id"] == MODEL_ID
assert config["version"] == MODEL_VERSION

training_module = run_path(str(train_path))
make_model = training_module["make_model"]
model = make_model(config)

print("Mô hình:", model)
display(config["params"])

Mô hình: DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)


{'criterion': 'gini',
 'max_depth': 5,
 'min_samples_leaf': 10,
 'random_state': 42}

## Mục 5 - Chọn run Decision Tree v1 đã lưu

In [10]:
RUN_ID = "m02_decision_tree_v1_20261104T095332_6d8b7c85"
run_path = FINAL / "experiments" / "runs" / RUN_ID

if not run_path.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục kết quả: {run_path}"
    )

print("Run:", RUN_ID)
print("Đường dẫn:", run_path.relative_to(FINAL).as_posix())

Run: m02_decision_tree_v1_20261104T095332_6d8b7c85
Đường dẫn: experiments/runs/m02_decision_tree_v1_20261104T095332_6d8b7c85
